In [ ]:
import os
import sys
import dask.dataframe as dd

module_path = os.path.abspath(os.path.join("."))
if module_path not in sys.path:
    sys.path.append(module_path)

from utilities import walk_through_files, save_df_in_chunks

from dask.distributed import Client

client = Client()  # Connect to distributed cluster and override default
client

RECORDS_PATH = os.path.join("raw-data", "sg_subzones_devices")
SAVEPATH = os.path.join("intermediate-files", "tt-appearances-areadate")

In [ ]:
file_lists = tuple(walk_through_files(RECORDS_PATH, "csv.gz", fullpath=True))
len(file_lists)

In [ ]:
%%time
ddf = (
    (
        dd.read_csv(
            file_lists,
            compression="gzip",
            usecols=[0],
            names=["deviceID"],
            blocksize=None,
        )
        .groupby("deviceID")
        .size()
        .reset_index()
        .rename(columns={0: "tt_appearances_areadate"})
        .astype({"tt_appearances_areadate": "int"})
    )
    .compute()
    .sort_values("tt_appearances_areadate", ascending=False, ignore_index=True)
)

In [ ]:
save_df_in_chunks(ddf, SAVEPATH, 100, "device-appearances-chunk", "csv.gz", "gzip")

In [ ]:
len(ddf[ddf["tt_appearances_areadate"] == 1])

In [ ]:
len(ddf[ddf["tt_appearances_areadate"] > 400])

In [ ]:
len(ddf[ddf["tt_appearances_areadate"] < 50])

In [ ]:
bins = [x / 100 for x in list(range(5, 100 + 5, 5))]

ddf.describe(percentiles=bins)